In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt  # for making figures
import random

%matplotlib inline

In [2]:
words = open("names.txt", "r").read().splitlines()
len(words)

32033

In [3]:
# shuffle up the words
random.seed(42)
random.shuffle(words)

In [4]:
# get distinct chars across all words -- build vocab
chars = sorted(list(set("".join(words))))

# map chars to and from integers
# 0th char will be "." which denotes start/end of a word
char_to_int = {ch: idx + 1 for idx, ch in enumerate(chars)}
char_to_int["."] = 0
int_to_char = {idx: ch for ch, idx in char_to_int.items()}

print(char_to_int)
print(int_to_char)

{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}
{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
# build train, eval and test datasets
# X: 3 chars --> Y: next char
# For word "emma"
# ... --> e
# ..e --> m
# .em --> m
# emm --> a
# mma --> .

block_size = 8  # number of chars we take to predict the next one


def get_dataset(words):
    X = []
    y = []
    for w in words:
        block = [0] * block_size  # we start a block with all "."
        # iterate over every character in the word plus "." to mark the end
        # plus "." is necessary or else the loop will completely skip last char of the word
        for ch in w + ".":
            ix = char_to_int[ch]
            y.append(ix)
            X.append(block)
            block = block[1:] + [ix]  # move the window forward
    X = torch.tensor(X)
    y = torch.tensor(y)
    return X, y


# ------------------------------
# Viz get_dataset output first
# ------------------------------
# X, y = get_dataset(words[:25])
# for xout, yout in zip(X, y):
#     xout_str = "".join([int_to_char[ch.item()] for ch in xout])
#     yout_str = int_to_char[yout.item()]
#     print("{} --> {}".format(xout_str, yout_str))

# ------------------------------
# train/val/test datasets
# ------------------------------
n1 = int(len(words) * 0.8)
n2 = int(len(words) * 0.9)

Xtr, Ytr = get_dataset(words[:n1])
Xval, Yval = get_dataset(words[n1:n2])
Xtest, Ytest = get_dataset(words[n2:])

print("Training set: X={}, y={}".format(Xtr.shape, Ytr.shape))
print("Validation set: X={}, y={}".format(Xval.shape, Yval.shape))
print("Test set: X={}, y={}".format(Xtest.shape, Ytest.shape))


Training set: X=torch.Size([182625, 8]), y=torch.Size([182625])
Validation set: X=torch.Size([22655, 8]), y=torch.Size([22655])
Test set: X=torch.Size([22866, 8]), y=torch.Size([22866])


#### Part 1: Semi-Torchified e2e architecture (no Wavenet)

In [ ]:
# ------------------------------------------------------------
# Rough outline of what we need
# i/p --> embedding + flattening --> linear 1 --> BN --> tanh --> linear 2 --> cross-entropy loss
# ------------------------------------------------------------

# g = torch.Generator().manual_seed(6)

# #
# vocab_size = len(chars) + 1
# block_size = 10  # already defined above; stating here for easy reference
# n_emb = 10
# n_hidden = 100

# # embedding layer; flattening would happen in training loop
# C = torch.randn((vocab_size, n_emb), generator=g)
# print(C.shape)

# # linear layer 1
# W1 = torch.randn((block_size * n_emb, n_hidden), generator=g)
# b1 = torch.zeros(n_hidden)

# # batchnorm parameter initialization
# # we want to start with gain = 1 and bias = 0. These params will be learned while training
# # in PyTorch nomenclature, bngain (gamma) and bnbias (beta) are referred to as "buffers"
# bngain = torch.ones((1, n_hidden))
# bnbias = torch.zeros((1, n_hidden))
# # running mean and std for running predictions post-training [no grad required]
# bnmean_running = torch.zeros((1, n_hidden))
# bnstd_running = torch.ones((1, n_hidden))

# # linear layer 2
# W2 = torch.randn((n_hidden, vocab_size), generator=g)
# b2 = torch.zeros(vocab_size, generator=g)

# parameters = [C, W1, W2, b2, bngain, bnbias]
# for p in parameters:
#     p.requires_grad = True


In [ ]:
# Build the required layers similar to PyTorch API

g = torch.Generator().manual_seed(6)


class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out), generator=g)
        self.weight /= fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, X):
        self.out = X @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])


class Tanh:
    def __call__(self, X):
        self.out = torch.tanh(X)
        return self.out

    def parameters(self):
        return []


# bn = gamma * (x - xmean())/(xstd() + eps) + beta --> when training
# bn = gamma * (x - x_running_mean)/x_running_std + beta --> when not training
class BatchNorm1d:
    def __init__(self, num_features, eps=1e-05, momentum=0.01):
        # batchnorm params
        self.num_features = num_features
        self.eps = eps
        self.gamma = torch.ones(num_features)  # batchnorm gain
        self.beta = torch.zeros(num_features)  # batchnorm bias
        # indicating batchnorm mode
        self.training = True
        # parameters for maintaining running mean and var
        self.x_running_mean = torch.zeros(num_features)
        self.x_running_var = torch.ones(num_features)
        self.momentum = momentum

    def __call__(self, X):
        # dim=0 because we calculate mean for each neuron across all examples
        if self.training:
            xmean = X.mean(0, keepdim=False)
            xvar = X.var(0, keepdim=False)
        else:
            xmean = self.x_running_mean
            xvar = self.x_running_var

        # perform batchnorm
        self.out = self.gamma * (X - xmean) / torch.sqrt(xvar + self.eps) + self.beta

        # maintain running mean and var
        if self.training:
            with torch.no_grad():
                self.x_running_mean = (1 - self.momentum) * self.x_running_mean + self.momentum * xmean
                self.x_running_var = (1 - self.momentum) * self.x_running_var + self.momentum * xvar

        return self.out

    def parameters(self):
        return [self.gamma, self.beta]


# ------------------------------------------------------------
# # dummy example to ensure layers are working
# a = Linear(block_size * n_emb, n_hidden, n_hidden)
# a_in = torch.randn(10, block_size * n_emb)
# a_out = a(a_in)
# b = BatchNorm1d(a_out.shape[1])
# b(a_out)

# print(a.weight.shape, a_in.shape, a_out.shape)
# print(a.parameters())
# print(a_out.shape, b.out.shape)
# print(b.parameters())
# ------------------------------------------------------------

In [ ]:
# Setup and initialize the network

vocab_size = len(chars) + 1
block_size = 8  # already defined above; stating here for easy reference
n_emb = 10
n_hidden = 100

# layers
C = torch.randn((vocab_size, n_emb))
layers = [
    Linear(fan_in=n_emb * block_size, fan_out=n_hidden, bias=False),  # bias false since we using BN next
    BatchNorm1d(num_features=n_hidden),
    Tanh(),
    Linear(fan_in=n_hidden, fan_out=n_hidden, bias=False),
    BatchNorm1d(num_features=n_hidden),
    Tanh(),
    Linear(fan_in=n_hidden, fan_out=vocab_size, bias=False),
    BatchNorm1d(num_features=vocab_size),
]


# additional initialization
with torch.no_grad():
    # (1) make last layer less confident
    layers[-1].gamma *= 0.1  # when not using batchnorm as last layer, use layers[-1].weight *= 0.1

    # (2) kaiming initialization for linear layers using tanh activation; only last layer doesnt use tanh
    for layer in layers[:-1]:
        if isinstance(layer, Linear):
            layer.weight *= 5 / 3


# consolidate all parameters
parameters = [C] + [p for layer in layers for p in layer.parameters()]


# set requires_grad true for all parameters
for p in parameters:
    p.requires_grad = True


# total parameters in network
network_nparams = 0
for layer in layers:
    layer_nparams = sum([p.nelement() for p in layer.parameters()])
    network_nparams += layer_nparams
    print("{} --> total params: {}".format(layer.__class__.__name__, layer_nparams))
print("Total network params: {}".format(network_nparams))

In [ ]:
# full training loop

num_iterations = 200000
batch_size = 30
train_lossi = []

for i in range(num_iterations):
    # ----------------------------------BATCHING---------------------------------------------------------
    batch_idx = torch.randint(low=0, high=Xtr.shape[0], size=(batch_size,), generator=g)
    Xb, Yb = Xtr[batch_idx], Ytr[batch_idx]

    # ----------------------------------FORWARD PASS-----------------------------------------------------
    # fwd pass thro embedding + flattening layers
    emb = C[Xb]
    x = emb.view(emb.shape[0], -1)

    # fwd pass thro remaining layers
    for layer in layers:
        x = layer(x)

    # loss
    loss = F.cross_entropy(x, Yb)

    # ----------------------------------BACKWARD PASS----------------------------------------------------
    # run backward pass after nulling the existing gradients
    for p in parameters:
        p.grad = None
    loss.backward()

    # backpropgate gradients
    lr = 0.1 if i < 150000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    # ----------------------------------LOGGING----------------------------------------------------------
    # # we track log10 of cross-entropy loss for easier viz
    train_lossi.append(loss.log10().item())

    if i % 1000 == 0:
        print("Iter: {} | Train Loss: {}".format(i, loss.item()))
    # break


In [ ]:
# Loss viz
def loss_visualization(train_loss):
    # visualize mean loss over "x" no. of iterations to obtain a less noisy graph
    plt.plot(torch.tensor(train_loss).view(-1, 1000).mean(dim=1))
    plt.show()


loss_visualization(train_lossi)

In [ ]:
# loss on train and val sets

for layer in layers:
    if isinstance(layer, BatchNorm1d):
        layer.training = False


def get_loss(data, labels, stage):
    emb = C[data]
    x = emb.view(emb.shape[0], -1)
    for layer in layers:
        x = layer(x)
    loss = F.cross_entropy(x, labels)
    print("Stage: {} | Loss: {}".format(stage, loss))
    return loss


training_loss = get_loss(data=Xtr, labels=Ytr, stage="training")
validation_loss = get_loss(data=Xval, labels=Yval, stage="validation")


In [ ]:
# use model to make predictions

total_names_to_generate = 20

for model_run in range(total_names_to_generate):
    out = []
    block = [0] * block_size

    while True:
        # passing one example through the network
        emb = C[block]
        x = emb.view(1, -1)
        for layer in layers:
            x = layer(x)
        # x represents logits at the end of network
        # obtain the probabilities using softmax --> probabilities for each of the 27 characters for our example
        probab = F.softmax(x, dim=1)
        # make prediction based on the probab distribution
        next_char = torch.multinomial(probab, num_samples=1, generator=g).item()
        # update context and store the predictions
        block = block[1:] + [next_char]
        out.append(next_char)
        # if we predict next char as ".", we mark that as end of word
        if next_char == 0:
            break

    print("".join([int_to_char[c] for c in out]))

#### Part 2: Fully-Torchified e2e architecture (no Wavenet)

In [6]:
torch.manual_seed(12346)  # seed rng for reproducibility

In [9]:
# Build the required layers similar to PyTorch API


class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out))
        self.weight /= fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, X):
        self.out = X @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])


class Tanh:
    def __call__(self, X):
        self.out = torch.tanh(X)
        return self.out

    def parameters(self):
        return []


# bn = gamma * (x - xmean())/(xstd() + eps) + beta --> when training
# bn = gamma * (x - x_running_mean)/x_running_std + beta --> when not training
class BatchNorm1d:
    def __init__(self, num_features, eps=1e-05, momentum=0.01):
        # batchnorm params
        self.num_features = num_features
        self.eps = eps
        self.gamma = torch.ones(num_features)  # batchnorm gain
        self.beta = torch.zeros(num_features)  # batchnorm bias
        # indicating batchnorm mode
        self.training = True
        # parameters for maintaining running mean and var
        self.x_running_mean = torch.zeros(num_features)
        self.x_running_var = torch.ones(num_features)
        self.momentum = momentum

    def __call__(self, X):
        # dim=0 because we calculate mean for each neuron across all examples
        if self.training:
            xmean = X.mean(0, keepdim=False)
            xvar = X.var(0, keepdim=False)
        else:
            xmean = self.x_running_mean
            xvar = self.x_running_var

        # perform batchnorm
        self.out = self.gamma * (X - xmean) / torch.sqrt(xvar + self.eps) + self.beta

        # maintain running mean and var
        if self.training:
            with torch.no_grad():
                self.x_running_mean = (1 - self.momentum) * self.x_running_mean + self.momentum * xmean
                self.x_running_var = (1 - self.momentum) * self.x_running_var + self.momentum * xvar

        return self.out

    def parameters(self):
        return [self.gamma, self.beta]


class Embedding:
    def __init__(self, num_embeddings, embedding_dim):
        # num_embeddings: size of the dictionary of embeddings
        # in our case: num_embeddings represent total unique characters (vocabulary) we want embeddings for
        # embedding_dim: the size of each embedding vector
        self.embeddings = torch.randn(num_embeddings, embedding_dim)

    def __call__(self, X):
        self.out = self.embeddings[X]
        return self.out

    def parameters(self):
        return [self.embeddings]


class Flattening:
    def __call__(self, X):
        self.out = X.view(X.shape[0], -1)
        return self.out

    def parameters(self):
        return []


# Container class
class Sequential:
    def __init__(self, layers):
        self.layers = layers

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        self.out = x
        return self.out

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


# ------------------------------------------------------------------------------------------------------------------------
# dummy example to ensure layers are configured properly

total_examples_dummy_p2 = 10
block_size_dummy_p2 = 8

ip_dummy_p2 = torch.randint(low=0, high=27, size=(total_examples_dummy_p2, block_size_dummy_p2))
print(f"Input data: {ip_dummy_p2.shape}")
print("-----------------------")

# initialize and use Emb Layer
vocab_size_dummy_p2 = 27
n_emb_dummy_p2 = 3

EmbLayer = Embedding(num_embeddings=vocab_size_dummy_p2, embedding_dim=n_emb_dummy_p2)
emb = EmbLayer(ip_dummy_p2)

print(f"Embedding layer --> Emb: {EmbLayer.embeddings.shape} | output: {emb.shape}")
print("Total params: {}".format(sum([p.nelement() for p in EmbLayer.parameters()])))
print("-----------------------")

# initialize and use Flattening Layer
FlatLayer = Flattening()
emb_flat = FlatLayer(emb)

print(f"Flattening layer --> output: {emb_flat.shape}")
print("Total params: {}".format(sum([p.nelement() for p in FlatLayer.parameters()])))
print("-----------------------")

# initialize and use the linear layer
n_hidden_dummy_p2 = 100

LinearLayer = Linear(fan_in=emb_flat.shape[1], fan_out=n_hidden_dummy_p2)
linear_out = LinearLayer(emb_flat)
print(f"Linear layer --> W: {LinearLayer.weight.shape} | b: {LinearLayer.bias.shape} | output: {linear_out.shape}")
print("Total params: {}".format(sum([p.nelement() for p in LinearLayer.parameters()])))
print("-----------------------")

# initialize and use the batchnorm layer
BNLayer = BatchNorm1d(num_features=n_hidden_dummy_p2)
BNout = BNLayer(linear_out)
print(
    f"BN layer --> gamma: {BNLayer.gamma.shape} | beta: {BNLayer.beta.shape} | running_mean: {BNLayer.x_running_mean.shape} | running_var: {BNLayer.x_running_var.shape} | output: {BNout.shape} "
)
print("Total params: {}".format(sum([p.nelement() for p in BNLayer.parameters()])))
print("-----------------------")

# initialize all the layers using sequential
container = Sequential(
    [
        Embedding(num_embeddings=vocab_size_dummy_p2, embedding_dim=n_emb_dummy_p2),
        Flattening(),
        Linear(fan_in=block_size_dummy_p2 * n_emb_dummy_p2, fan_out=n_hidden_dummy_p2),
        BatchNorm1d(num_features=n_hidden_dummy_p2),
        Tanh(),
    ]
)

container_out = container.forward(ip_dummy_p2)
print(f"Container --> input data: {ip_dummy_p2.shape} | output: {container_out.shape}")
print("Total params: {}".format(sum([p.nelement() for p in container.parameters()])))

for i in range(len(container.layers)):
    print(
        f"Layer No. = {i} | Layer = {container.layers[i].__class__.__name__} | Output = {container.layers[i].out.shape}"
    )
print("-----------------------")

Input data: torch.Size([10, 8])
-----------------------
Embedding layer --> Emb: torch.Size([27, 3]) | output: torch.Size([10, 8, 3])
Total params: 81
-----------------------
Flattening layer --> output: torch.Size([10, 24])
Total params: 0
-----------------------
Linear layer --> W: torch.Size([24, 100]) | b: torch.Size([100]) | output: torch.Size([10, 100])
Total params: 2500
-----------------------
BN layer --> gamma: torch.Size([100]) | beta: torch.Size([100]) | running_mean: torch.Size([100]) | running_var: torch.Size([100]) | output: torch.Size([10, 100]) 
Total params: 200
-----------------------
Container --> input data: torch.Size([10, 8]) | output: torch.Size([10, 100])
Total params: 2781
Layer No. = 0 | Layer = Embedding | Output = torch.Size([10, 8, 3])
Layer No. = 1 | Layer = Flattening | Output = torch.Size([10, 24])
Layer No. = 2 | Layer = Linear | Output = torch.Size([10, 100])
Layer No. = 3 | Layer = BatchNorm1d | Output = torch.Size([10, 100])
Layer No. = 4 | Layer = 

In [10]:
# Setup and initialize the network

vocab_size = len(chars) + 1
block_size = 8  # already defined above; stating here for easy reference
n_emb = 10  # embedding dimensions; number of dimensions we want every unique entry to have
n_hidden = 100

# layers
model = Sequential(
    [
        Embedding(num_embeddings=vocab_size, embedding_dim=n_emb),
        Flattening(),
        Linear(fan_in=n_emb * block_size, fan_out=n_hidden, bias=False),  # bias false since we using BN next
        BatchNorm1d(num_features=n_hidden),
        Tanh(),
        Linear(fan_in=n_hidden, fan_out=n_hidden, bias=False),
        BatchNorm1d(num_features=n_hidden),
        Tanh(),
        Linear(fan_in=n_hidden, fan_out=vocab_size, bias=False),
        BatchNorm1d(num_features=vocab_size),
    ]
)


# additional initialization
with torch.no_grad():
    # (1) make last layer less confident
    model.layers[-1].gamma *= 0.1  # when not using batchnorm as last layer, use layers[-1].weight *= 0.1

    # (2) kaiming initialization for linear layers using tanh activation; only last layer doesnt use tanh
    for layer in model.layers[:-1]:
        if isinstance(layer, Linear):
            layer.weight *= 5 / 3

# get all model parameters
parameters = model.parameters()

# set requires_grad true for all parameters and confirm the update
for p in parameters:
    p.requires_grad = True
print(f"All parameters have requires_grad turned on --> {all(p.requires_grad for p in model.parameters())}")


# total parameters in network
network_nparams = 0
for layer in model.layers:
    layer_nparams = sum([p.nelement() for p in layer.parameters()])
    network_nparams += layer_nparams
    print("{} --> total params: {}".format(layer.__class__.__name__, layer_nparams))
print("Total network params: {}".format(network_nparams))

All parameters have requires_grad turned on --> True
Embedding --> total params: 270
Flattening --> total params: 0
Linear --> total params: 8000
BatchNorm1d --> total params: 200
Tanh --> total params: 0
Linear --> total params: 10000
BatchNorm1d --> total params: 200
Tanh --> total params: 0
Linear --> total params: 2700
BatchNorm1d --> total params: 54
Total network params: 21424


In [ ]:
# full training loop

num_iterations = 200000
batch_size = 30
train_lossi = []

for i in range(num_iterations):
    # ----------------------------------BATCHING---------------------------------------------------------
    batch_idx = torch.randint(low=0, high=Xtr.shape[0], size=(batch_size,))
    Xb, Yb = Xtr[batch_idx], Ytr[batch_idx]

    # ----------------------------------FORWARD PASS-----------------------------------------------------
    # fwd pass thro entire model + loss
    logits = model.forward(Xb)
    loss = F.cross_entropy(logits, Yb)

    # ----------------------------------BACKWARD PASS----------------------------------------------------
    # run backward pass after nulling the existing gradients
    for p in parameters:
        p.grad = None
    loss.backward()

    # backpropgate gradients
    lr = 0.1 if i < 150000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    # ----------------------------------LOGGING----------------------------------------------------------
    # # we track log10 of cross-entropy loss for easier viz
    train_lossi.append(loss.log10().item())

    if i % 1000 == 0:
        print("Iter: {} | Train Loss: {}".format(i, loss.item()))
    # break


Iter: 0 | Train Loss: 3.2640023231506348


In [12]:
# Loss viz
def loss_visualization(train_loss):
    # visualize mean loss over "x" no. of iterations to obtain a less noisy graph
    plt.plot(torch.tensor(train_loss).view(-1, 1000).mean(dim=1))
    plt.show()


loss_visualization(train_lossi)

RuntimeError: shape '[-1, 1000]' is invalid for input of size 1

In [13]:
# loss on train and val sets

for layer in model.layers:
    if isinstance(layer, BatchNorm1d):
        layer.training = False


@torch.no_grad()
def get_loss(data, labels, stage):
    logits = model.forward(data)
    loss = F.cross_entropy(logits, labels)
    print("Stage: {} | Loss: {}".format(stage, loss))
    return loss


training_loss = get_loss(data=Xtr, labels=Ytr, stage="training")
validation_loss = get_loss(data=Xval, labels=Yval, stage="validation")


Stage: training | Loss: 3.312093734741211
Stage: validation | Loss: 3.3129231929779053


In [14]:
# use model to make predictions

total_names_to_generate = 20

for model_run in range(total_names_to_generate):
    out = []
    block = [0] * block_size

    while True:
        # passing one example through the network
        logits = model.forward(torch.tensor([block]))
        # obtain the probabilities using softmax --> probabilities for each of the 27 characters for our example
        probab = F.softmax(logits, dim=1)
        # make prediction based on the probab distribution
        next_char = torch.multinomial(probab, num_samples=1).item()
        # update context and store the predictions
        block = block[1:] + [next_char]
        out.append(next_char)
        # if we predict next char as ".", we mark that as end of word
        if next_char == 0:
            break

    print("".join([int_to_char[c] for c in out]))

jxybtcg.
kqiqitzmze.
tqhbbczusvvgqbbbqardr.
mjsuhrtlcmmmu.
kalskbbxyo.
w.
zvaupzqncpdgpqhfmzkgzirjeiwmu.
xqkbvqo.
lvxdklonjjjaclcbbxlkurqqjyplxxhxyuwnse.
yglqbqqkjkzojnzzcsccvaqehqsiiwvzdykmdm.
yivorszcibqmrqtoxful.
vhryityjgrmgfbc.
jnngnrmjvtmuvafvkkdoyvtyioxcoge.
lubzrworhydzuuvvzfdpoxnzhwrqkxcmdgpsb.
cchrduavbzfvswrxsxgmuyohcprqqmnymfldryvyennlrfzcdrtnryl.
mtbfpua.
oewylvoaahqomfrkrlrihaxulmxdwywy.
stfgavm.
tfhogiqyftynfagoztqckuztmcatmrjcqhvci.
pjryjecqqtlwxwxeucoctfqyqvbhidhmheaw.


#### Part 3: Fully-Torchified e2e architecture (Wavenet)

In [ ]:
torch.manual_seed(12346)  # seed rng for reproducibility

In [ ]:
# Build the required layers similar to PyTorch API


class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out))
        self.weight /= fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, X):
        self.out = X @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])


class Tanh:
    def __call__(self, X):
        self.out = torch.tanh(X)
        return self.out

    def parameters(self):
        return []


# bn = gamma * (x - xmean())/(xstd() + eps) + beta --> when training
# bn = gamma * (x - x_running_mean)/x_running_std + beta --> when not training
# our orig BatchNorm1d assumes X is 2-D. But for Wavenet, we need to update the xmean and xvar
# computations to accomodate third dimension (read notes for more details)
class BatchNorm1d:
    def __init__(self, num_features, eps=1e-05, momentum=0.01):
        # batchnorm params
        self.num_features = num_features
        self.eps = eps
        self.gamma = torch.ones(num_features)  # batchnorm gain
        self.beta = torch.zeros(num_features)  # batchnorm bias
        # indicating batchnorm mode
        self.training = True
        # parameters for maintaining running mean and var
        self.x_running_mean = torch.zeros(num_features)
        self.x_running_var = torch.ones(num_features)
        self.momentum = momentum

    def __call__(self, X):
        # dim=0 because we calculate mean for each neuron across all examples
        if self.training:
            if x.ndim == 2:
                batched_dim = 0
                xmean = X.mean(0, keepdim=False)
                xvar = X.var(0, keepdim=False)
            elif x.ndim == 3:
                batched_dim = (0, 1)
            xmean = X.mean(batched_dim, keepdim=True)
            xvar = X.var(batched_dim, keepdim=True)
        else:
            xmean = self.x_running_mean
            xvar = self.x_running_var

        # perform batchnorm
        self.out = self.gamma * (X - xmean) / torch.sqrt(xvar + self.eps) + self.beta

        # maintain running mean and var
        if self.training:
            with torch.no_grad():
                self.x_running_mean = (1 - self.momentum) * self.x_running_mean + self.momentum * xmean
                self.x_running_var = (1 - self.momentum) * self.x_running_var + self.momentum * xvar

        return self.out

    def parameters(self):
        return [self.gamma, self.beta]


class Embedding:
    def __init__(self, num_embeddings, embedding_dim):
        # num_embeddings: size of the dictionary of embeddings
        # in our case: num_embeddings represent total unique characters (vocabulary) we want embeddings for
        # embedding_dim: the size of each embedding vector
        self.embeddings = torch.randn(num_embeddings, embedding_dim)

    def __call__(self, X):
        self.out = self.embeddings[X]
        return self.out

    def parameters(self):
        return [self.embeddings]


# (1) FlattenConsecutive layer -- deviates from Torch's Flattening layer in terms of functioning and attributes
# (2) given (4, 8, 10) we want --> (4, 4, 20).
# meaning: flatten consecutive groups of 2 or flatten by a factor of 2 --> initialize using FlattenConsecutive(2)
# (3) in this implementation, the wavenet architecture only impacts 1st and 2nd dimensions (not 0th)
class FlattenConsecutive:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, X):
        # input will come from embedding layer and will have 3 dimensions in this implementation
        N, L, C = X.shape
        self.out = X.view(N, L // self.factor, C * self.factor)
        # if factor
        if self.factor == L:
            self.out = torch.squeeze(self.out)
        return self.out

    def parameters(self):
        return []


# Container class
class Sequential:
    def __init__(self, layers):
        self.layers = layers

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        self.out = x
        return self.out

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


# -------------------------------------------DUMMY EXAMPLE-------------------------------------------------
# dummy example to ensure layers are configured properly
vocab_size_dummy = 27
embedding_dim_dummy = 3  # this code interchangeably uses n_emb variable name for it. Need to make it consistent. TBD.
total_examples_dummy = 10
block_size_dummy = 8
n_hidden_dummy = 100

# initialize all the layers using sequential
container = Sequential(
    [
        Embedding(num_embeddings=vocab_size_dummy, embedding_dim=embedding_dim_dummy),
        # wavenet level 0
        FlattenConsecutive(factor=2),
        Linear(fan_in=embedding_dim_dummy * 2, fan_out=n_hidden_dummy, bias=False),
        BatchNorm1d(num_features=n_hidden_dummy),
        Tanh(),
        # wavenet level 1
        FlattenConsecutive(factor=2),
        Linear(fan_in=n_hidden_dummy * 2, fan_out=n_hidden_dummy, bias=False),
        BatchNorm1d(num_features=n_hidden_dummy),
        Tanh(),
        # wavenet level 2
        FlattenConsecutive(factor=2),
        Linear(fan_in=n_hidden_dummy * 2, fan_out=n_hidden_dummy, bias=False),
        BatchNorm1d(num_features=n_hidden_dummy),
        Tanh(),
        #
        Linear(fan_in=n_hidden_dummy, fan_out=vocab_size_dummy),
    ]
)

# dummy input
ip_dummy = torch.randint(low=0, high=vocab_size_dummy, size=(total_examples_dummy, block_size_dummy))
# run forward pass on model
container_out = container.forward(ip_dummy)

print(f"Container overall --> input data: {ip_dummy.shape} | output: {container_out.shape}")
print("Total params: {}".format(sum([p.nelement() for p in container.parameters()])))

print("-----------------------")
for i, layer in enumerate(container.layers):
    print(f"Layer No. = {i}\t| Layer = {layer.__class__.__name__}\t| Output = {layer.out.shape}".expandtabs(30))
print("-----------------------")

In [ ]:
# Setup and initialize the network

vocab_size = len(chars) + 1
block_size = 8  # already defined above; stating here for easy reference
n_emb = 10  # embedding dimensions; number of dimensions we want every unique entry to have
n_hidden = 200

# containerize all layers into a container class -- Sequential
model = Sequential(
    [
        Embedding(num_embeddings=vocab_size, embedding_dim=n_emb),
        # wavenet level 0
        FlattenConsecutive(2),
        Linear(fan_in=n_emb * 2, fan_out=n_hidden, bias=False),  # bias false since we using BN next
        BatchNorm1d(num_features=n_hidden),
        Tanh(),
        # wavenet level 1
        FlattenConsecutive(2),
        Linear(fan_in=n_hidden * 2, fan_out=n_hidden, bias=False),  # bias false since we using BN next
        BatchNorm1d(num_features=n_hidden),
        Tanh(),
        # wavenet level 2
        FlattenConsecutive(2),
        Linear(fan_in=n_hidden * 2, fan_out=n_hidden, bias=False),  # bias false since we using BN next
        BatchNorm1d(num_features=n_hidden),
        Tanh(),
        # final level
        Linear(fan_in=n_hidden, fan_out=vocab_size, bias=False),
    ]
)

# -------------------------------------------CONFIRM NETWORK DIMENSIONS & PARAMS------------------------------------------------
# # [COMMENT IT OUT AFTER TESTING]

# total_examples = 10
# ip = torch.randint(low=0, high=vocab_size, size=(total_examples_dummy, block_size))
# model.forward(ip)

# network_nparams = 0
# for i, layer in enumerate(model.layers):
#     layer_nparams = sum([p.nelement() for p in layer.parameters()])
#     network_nparams += layer_nparams
#     print(
#         f"Layer No. = {i}\t| Layer = {layer.__class__.__name__}\t| Output = {layer.out.shape}\t| params = {layer_nparams}".expandtabs(
#             30
#         )
#     )

# ------------------------------------------ADDITIONAL INITIALIZATIONS----------------------------------------------------------
# additional initialization
with torch.no_grad():
    # make last layer less confident
    model.layers[-1].weight *= 0.1  # since we are only using linear layer as last layer

# get all model parameters
parameters = model.parameters()
print(sum(p.nelement() for p in parameters))

# set requires_grad true for all parameters and confirm the update
for p in parameters:
    p.requires_grad = True
print(f"All parameters have requires_grad turned on --> {all(p.requires_grad for p in model.parameters())}")


In [ ]:
# full training loop

num_iterations = 1000  # 200000
batch_size = 30
train_lossi = []

for i in range(num_iterations):
    # ----------------------------------BATCHING---------------------------------------------------------
    batch_idx = torch.randint(low=0, high=Xtr.shape[0], size=(batch_size,))
    Xb, Yb = Xtr[batch_idx], Ytr[batch_idx]

    # ----------------------------------FORWARD PASS-----------------------------------------------------
    # fwd pass thro entire model + loss
    logits = model.forward(Xb)
    loss = F.cross_entropy(logits, Yb)

    # ----------------------------------BACKWARD PASS----------------------------------------------------
    # run backward pass after nulling the existing gradients
    for p in parameters:
        p.grad = None
    loss.backward()

    # backpropgate gradients
    lr = 0.1 if i < 150000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    # ----------------------------------LOGGING----------------------------------------------------------
    # # we track log10 of cross-entropy loss for easier viz
    train_lossi.append(loss.log10().item())

    if i % 1000 == 0:
        print("Iter: {} | Train Loss: {}".format(i, loss.item()))
    # break


In [ ]:
# Loss viz
def loss_visualization(train_loss):
    # visualize mean loss over "x" no. of iterations to obtain a less noisy graph
    plt.plot(torch.tensor(train_loss).view(-1, 1000).mean(dim=1))
    plt.show()


loss_visualization(train_lossi)

In [ ]:
# loss on train and val sets

for layer in model.layers:
    if isinstance(layer, BatchNorm1d):
        layer.training = False


@torch.no_grad()
def get_loss(data, labels, stage):
    logits = model.forward(data)
    loss = F.cross_entropy(logits, labels)
    print("Stage: {} | Loss: {}".format(stage, loss))
    return loss


training_loss = get_loss(data=Xtr, labels=Ytr, stage="training")
validation_loss = get_loss(data=Xval, labels=Yval, stage="validation")


In [ ]:
# use model to make predictions

total_names_to_generate = 20

for model_run in range(total_names_to_generate):
    out = []
    block = [0] * block_size

    while True:
        # passing one example through the network
        logits = model.forward(torch.tensor([block]))
        # obtain the probabilities using softmax --> probabilities for each of the 27 characters for our example
        probab = F.softmax(logits, dim=1)
        # make prediction based on the probab distribution
        next_char = torch.multinomial(probab, num_samples=1).item()
        # update context and store the predictions
        block = block[1:] + [next_char]
        out.append(next_char)
        # if we predict next char as ".", we mark that as end of word
        if next_char == 0:
            break

    print("".join([int_to_char[c] for c in out]))

In [ ]:
logits.shape